# Classifcation + Cross-validation + Threshold Tuning + Smote(optional)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE  # Optional

import warnings
warnings.filterwarnings("ignore")

# ----------------------------- 1. Load and Prepare Data -----------------------------
df = pd.read_csv("heart_disease.csv")  # Replace with your path
X = df.drop("target", axis=1)
y = df["target"]

# OPTIONAL: Apply SMOTE (for testing impact of imbalance handling)
apply_smote = False  # Change to True to enable SMOTE

if apply_smote:
    smote = SMOTE(random_state=42)
    X, y = smote.fit_resample(X, y)
    print("SMOTE applied. New class distribution:\n", y.value_counts())

# Split for threshold tuning
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# ----------------------------- 2. Define Models -----------------------------
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Random Forest": RandomForestClassifier(class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced"),
    "SVM": SVC(probability=True, class_weight="balanced"),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "AdaBoost": AdaBoostClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss"),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# ----------------------------- 3. Evaluation Function -----------------------------
def evaluate_classifier(model, X, y, cv=5):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    
    metrics = {
        "Accuracy": cross_val_score(model, X, y, cv=skf, scoring='accuracy').mean(),
        "Precision": cross_val_score(model, X, y, cv=skf, scoring='precision').mean(),
        "Recall": cross_val_score(model, X, y, cv=skf, scoring='recall').mean(),
        "F1 Score": cross_val_score(model, X, y, cv=skf, scoring='f1').mean(),
        "ROC AUC": cross_val_score(model, X, y, cv=skf, scoring='roc_auc').mean(),
    }
    
    return metrics

# ----------------------------- 4. Run Models -----------------------------
model_names = []
results = []

for name, model in models.items():
    print(f"\n🔍 Evaluating: {name}")
    scores = evaluate_classifier(model, X, y)
    
    for metric, value in scores.items():
        print(f"{metric}: {value:.4f}")
    
    model_names.append(name)
    results.append(scores)

# ----------------------------- 5. Threshold Tuning -----------------------------
def threshold_tuning(model, X_train, y_train, X_test, y_test, thresholds=np.arange(0.1, 0.9, 0.05)):
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]

    print("\n🎯 Threshold Tuning Results:")
    for thresh in thresholds:
        y_pred_thresh = (y_proba >= thresh).astype(int)
        precision = precision_score(y_test, y_pred_thresh)
        recall = recall_score(y_test, y_pred_thresh)
        f1 = f1_score(y_test, y_pred_thresh)
        print(f"Threshold: {thresh:.2f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    
    return y_proba

# Apply threshold tuning on key models
print("\n================= Threshold Tuning: Logistic Regression =================")
threshold_tuning(LogisticRegression(class_weight="balanced", max_iter=1000), X_train, y_train, X_test, y_test)

print("\n================= Threshold Tuning: XGBoost =================")
threshold_tuning(XGBClassifier(use_label_encoder=False, eval_metric="logloss"), X_train, y_train, X_test, y_test)

# ----------------------------- 6. Compile Results -----------------------------
results_df = pd.DataFrame(results, index=model_names)
results_df = results_df.sort_values(by="ROC AUC", ascending=False)

print("\n📊 Final Model Comparison (Sorted by ROC AUC):")
print(results_df)
